# A tour of ercot-mis

> **This notebook pulls ERCOT Secure / ECEII data into `demo/data/` (gitignored) and shows it on your
> screen.** Node, substation and equipment names are confidential: do not paste outputs anywhere, do not
> screenshot them, and strip the outputs before you commit this file. The pre-commit guard refuses a
> notebook with outputs. Strip them with:
>
>     uv run --with nbstripout nbstripout demo/tour.ipynb

Run it with `uv run --with jupyterlab jupyter lab demo/tour.ipynb` from the repository root, after
`uv sync` and with your `.env` and certificate in place (see CLAUDE.md, *Environment*). Every cell is a
call you would make yourself; the text between cells says what the call does and why the repository
does it that way. It fetches about 100 MB the first time and is cheap after that.

**What ercot-mis is.** A fetcher: it downloads ERCOT's network models and the inputs around them,
keeps the original bytes, turns them into typed tables, and standardizes them so that every table can
be traced back to the bytes ERCOT published. Analysis lives elsewhere; this repository's job is to
reproduce ERCOT's own numbers faithfully and to record every place it had to decide something.

**Words used below** (ERCOT's, with the repository's precise meaning):

- **node**: the finest connection point a model gives, one record of a PSS/E RAW file. PSS/E calls it a bus.
- **bus**: nodes joined by closed breakers, identified by `bus_key`. A DAM node is a bus on its own.
- **substation**: the physical yard, holding buses at several voltage levels.
- **settlement point**: a place ERCOT prices and settles at: a resource node (one bus), a hub (an
  average over hub buses) or a load zone (a load-weighted average over its loads' buses).
- **contingency**: a what-if outage the market secures against (a line, a transformer, sometimes a
  bus split in two).
- **shift factor**: how much of one MW injected at a node flows on a given branch. The market's
  prices are built from these.
- **shadow price**: what one more MW of room on a binding constraint would have been worth.
- **GTC** (generic transmission constraint): a limit ERCOT places on a group of lines together.
- **EMIL ID** (`NP4-500-SG` and the like): ERCOT's catalogue number for a data product
  (EMIL is ERCOT's Market Information List).
- **PSS/E** and **RAW file**: a commercial power-flow program and its text case format;
  both ERCOT models are shipped as PSS/E RAW files.
- **DC power flow**: the linear approximation of how power splits over lines, using only
  each line's reactance; ERCOT's market engines price with it, and so does this repository.
- **slack**: the one node a DC solve measures everything against; it does not change
  flows, only which node has a sensitivity of zero.
- **tie**: in the CRR model, a closed breaker or switch written as a line at PSS/E's
  minimum reactance; the auction solves them as branches.
- **electrical bus**: ERCOT's name for a pricing point in its price and mapping files;
  only the raw and mapping tables use the term.
- **time-of-use block** (`PeakWD`, `PeakWE`, `Off-peak`): the CRR auction's hour groups;
  CRR ratings are given per block.
- **GTL**: the daily workbook of GTC limits; GTC is the constraint, GTL its limit.
- **DC tie**: a converter station linking ERCOT to a neighbouring grid; a settlement point of its own kind.
- **shift factor** (also called PTDF): see above; a row of them is the sensitivity of
  one branch to every node.

In [ ]:
from datetime import date, timedelta
from pathlib import Path

import polars as pl

import ercot_mis as em

pl.Config.set_tbl_rows(30)
REPO = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA = REPO / "demo" / "data"  # a data folder of its own, so the tour never touches data/
s = em.open(DATA)
s.data_dir

## 1. Products: what ERCOT publishes and which of it we take

`em.PRODUCTS` is the list of ERCOT data products the repository knows. Two **sources** deliver them:

- **EWS** (External Web Service): ERCOT's certificate-authenticated service for market participants.
  It carries the confidential network models. It keeps nothing older than each product's **display
  window** (31 days for DAM models, a year for CRR models): what is not captured in time is gone,
  which is why a scheduled daily pull exists.
- **Public API**: ERCOT's public archive. It keeps years, so a gap there heals itself.

Each product is either **pulled** (downloaded) or **tracked** (only listed, so its availability is on
record). `classification` says who may see it: Public, Secure, or ECEII (critical energy infrastructure
information, the strictest).

In [ ]:
pl.DataFrame([{"emil_id": p.emil_id, "name": p.name, "classification": p.classification, "source": p.source,
               "take": p.take, "window_days": p.display_days} for p in em.PRODUCTS.values()])

## 2. Listing and the archive

`list` asks ERCOT what it currently offers for a product and records every document in the local
**catalog** (a DuckDB file). It never returns ERCOT's file names, because those embed the participant's
DUNS number. `fetch` downloads what is listed and not archived yet.

The **archive** keeps the original zips, named by the SHA-256 of their bytes, read-only, never
unzipped to disk. A re-posted document (`_Upd`) is a new document, not an overwrite. The catalog
remembers every listing, every blob, every zip member and, later, every table built from them: this is
the provenance chain from a row back to ERCOT's bytes.

In [ ]:
docs = s.list("NP7-800-M")  # the monthly CRR network model
docs.select("doc_id", "posted_at", "size_bytes", "is_archived").tail(5)

The fetch below takes one DAM day (24 hourly models, about 30 MB), the monthly CRR model posted most
recently (about 20 MB), the GTC limits workbook, ERCOT's settlement point mappings, the day's published
prices and pricing inputs, and one 60-day disclosure document. `operating_dates` restricts a DAM fetch
to the posting days you want (the package posted on a day holds the next operating day's models); `since` bounds the posting time. Each call returns one row per document with its
status; a failed download is retried on the next call.

In [ ]:
DAY = date.today() - timedelta(days=1)  # the DAM package posted yesterday holds today's 24 hourly models
pulls = {
    "NP4-500-SG": dict(operating_dates=[DAY]),
    "NP7-800-M": dict(since=DAY - timedelta(days=45)),
    "NP3-766-M": dict(since=DAY - timedelta(days=2)),
    "NP3-220-SG": dict(since=DAY - timedelta(days=20)),
    "NP4-160-SG": dict(since=DAY - timedelta(days=20)),
    "NP3-966-ER": dict(since=DAY - timedelta(days=1)),
}
for emil_id in ("NP4-191-CD", "NP4-183-CD", "NP4-190-CD", "NP4-523-CD", "NP4-200-CD", "NP4-158-SG", "NP4-231-CD"):
    pulls[emil_id] = dict(since=DAY - timedelta(days=1))
for emil_id, kwargs in pulls.items():
    result = s.fetch(emil_id, **kwargs)
    print(f"{emil_id:11s}", dict(result.group_by("status").len().rows()), f"{result['size_bytes'].sum() / 1e6:.1f} MB")

## 3. The raw layer: one typed table per source file

`build_raw` parses every archived package of a product into Parquet tables under `raw/`, one table per
kind of source file, named for the file (`psse_bus`, `dam_lines`, `crr_contingencies`, ...). Column names
are made snake_case and values are typed; nothing else changes. Every row carries identity columns
(`emil_id`, `doc_id`, `blob_sha256`, `member_path`, and the package's own keys such as `operating_date`
and `hour`), and the catalog records each written file as an **artifact** keyed by the parser version
and the blob, so a rerun skips what is already built and a parser change rebuilds exactly what it
produced.

The PSS/E RAW files (the network itself) are read by the repository's own parser, which handles the
two dialects ERCOT uses (CRR and DAM). Everything else is CSV or a workbook, read with an exact
header check so that a column ERCOT adds or renames is noticed rather than guessed around.

In [ ]:
for emil_id in pulls:
    result = s.build_raw(emil_id)
    built = result.filter(pl.col("status") == "built")
    print(f"{emil_id:11s}", dict(result.group_by("status").len().rows()), f"{int(built['rows'].sum() or 0):,} rows")

In [ ]:
print(sorted(p.name for p in (s.data_dir / "raw").iterdir() if not p.name.startswith(".")))
s.raw("psse_bus").group_by("emil_id").len().collect()

## 4. The core layer: tidy tables shared by CRR and DAM

`build_core` turns each model into the same set of keyed tables, so the CRR and DAM models can be
compared row for row. Every row carries a `snapshot_id`: one per model, with a revision number
(`crr:monthly:2026-10:r1`, `dam:2026-10-05:he12:r1`). The DAM posts 24 models a day, one per hour,
and renumbers its nodes every hour, so the hour is part of the id.

- `node`: one row per node with its **`bus_key`**, an identity built from the equipment attached to the
  node rather than from PSS/E numbers (which the DAM changes hourly) or names (which are substation
  names in the DAM). In the CRR model, nodes joined by closed breakers share a `bus_key`.
- `branch` and `branch_rating`: lines and transformers with a stable `branch_id`, and both rating
  sources side by side (the RAW's rate A/B/C and the CRR's monitored-element ratings per time-of-use block).
- `contingency` and `contingency_outage`: what each contingency removes, resolved to branch and bus
  keys; a split-bus row records which end of a branch moves.
- `gtc` and `gtc_member`: the group constraints (CRR gives members; the DAM gives daily limits).
- `settlement_point` and `settlement_point_bus`: each settlement point's buses and weights. A DAM hub
  follows the protocols: equal weight per hub bus, then per energized bus inside it.
- `load`: every load with its node, service status and MW.

Where the code had to decide something, the decision has a row in `docs/assumptions.md` (the
**register**) and the code cites that row; a test fails when it does not.

In [ ]:
result = s.build_core()
print(dict(result.group_by("status").len().rows()))
snap = s.core("snapshot").collect()
snap.group_by("model_kind").agg(pl.len().alias("snapshots"), pl.col("snapshot_id").min().alias("first"), pl.col("snapshot_id").max().alias("last")).sort("model_kind")

In [ ]:
dam = snap.filter((pl.col("model_kind") == "dam") & (pl.col("hour") == 12)).sort("operating_date", "revision")
DAM_ID = dam["snapshot_id"][-1]  # hour 12 of the newest DAM day fetched
DAY = dam["operating_date"][-1]
monthly = snap.filter(pl.col("model_kind") == "monthly").sort("month", "revision")
same_month = monthly.filter(pl.col("month") == DAY.replace(day=1))
CRR_ID = (same_month if same_month.height else monthly)["snapshot_id"][-1]
print("DAM snapshot:", DAM_ID, "| CRR snapshot:", CRR_ID)

def counts(snapshot_id):
    node = s.core("node").filter(pl.col("snapshot_id") == snapshot_id).collect()
    return {
        "nodes": node.height, "buses": node["bus_key"].n_unique(),
        "branches": s.core("branch").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),
        "contingencies": s.core("contingency").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),
        "gtcs": s.core("gtc").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),
        "settlement_points": dict(s.core("settlement_point").filter(pl.col("snapshot_id") == snapshot_id).group_by("kind").len().sort("kind").collect().rows()),
    }
pl.DataFrame([{"snapshot": "CRR", **counts(CRR_ID)}, {"snapshot": "DAM", **counts(DAM_ID)}])

## 5. Matching the two models, and recording where they differ

The CRR model (used to auction congestion revenue rights months ahead) and the DAM model (used to clear
the day-ahead market) describe the same grid in different vocabularies. `match_*` records identity;
`diff_*` records differences, and the repository never smooths a difference over.

The order was settled by measurement: settlement points and generator or load names first, then
branches by the CRR workbook's operations name with a documented normalization, then buses through
the endpoints of matched branches (never by name or number), then contingencies by name with their
member sets compared. Every match carries its `match_method`; unmatched records are kept in the output.

In [ ]:
mb = s.match_branches(CRR_ID, DAM_ID)
print("branches by match method:", dict(mb.group_by("match_method").len().sort("match_method").rows()))
mbus = s.match_buses(CRR_ID, DAM_ID)
print("buses by match method:", dict(mbus.group_by("match_method").len().sort("match_method").rows()))
mc = s.match_contingencies(CRR_ID, DAM_ID)
print("contingencies by match method:", dict(mc.group_by("match_method").len().sort("match_method").rows()))

In [ ]:
diff = s.diff_branches(CRR_ID, DAM_ID)
verdicts = [c for c in diff.columns if c.startswith("same_")]
print("matched branches compared:", diff.height)
diff.select([pl.col(c).mean().round(3) for c in verdicts])  # share of matched branches that agree, per aspect

In [ ]:
print("settlement point rows compared:", s.diff_settlement_points(CRR_ID, DAM_ID).height)
print("load rows compared:", s.diff_loads(CRR_ID, DAM_ID).height)

## 6. The network: one snapshot as a solver reads it

`network` assembles one snapshot into the shape a DC power-flow solver wants: indexed nodes, branches
with limits, contingencies as index sets, signed GTC members and settlement point weights, plus a list
of everything it dropped and why. Each reason names a register row.

**The defaults do what ERCOT does.** The CRR auction solves the CRR model with its closed breakers as
branches, so the network keeps them; contracting them into buses is a named option
(`Options(contract_ties=True)`). The same goes for which rating is the limit and whether out-of-service
elements are kept. Networks are cached on disk under `out/network/` by snapshot and option set.

In [ ]:
from ercot_mis.out.network import Options

net = s.network(DAM_ID)
print(net.summary())
print(Options())

## 7. Shift factors and the test against what cleared

`ercot_mis.shift_factors.DcSystem` is the one place shift factors are computed: it factorizes the
network once and re-solves it under a contingency (removing branches, splitting buses, handling the
part of the grid a contingency cuts off).

The repository tests its models against ERCOT's published results **without re-clearing the market**.
The day-ahead market publishes the constraints that bound and their shadow prices, and it publishes the
prices. ERCOT's price at a node is the system price minus, for every binding constraint, the shadow
price times the node's shift factor on it (there is no loss term in the DAM). So if our network, our
names and our contingency definitions are right, we can rebuild every settlement point price from the
published shadow prices alone. The leftover, the **residual**, is the measure. `scripts/check_prices.py`
does this per hour and writes `data/reports/prices/<snapshot>.json`; the daily pull runs it on the
newest days. Numbers live in those reports, never in the documentation.

In [ ]:
from ercot_mis.shift_factors import DcSystem

system = DcSystem(net)
row = system.shift_factors([0])[0]  # how much of one MW at each node flows on branch 0
print("nodes:", row.size, "| largest shift factor on branch 0:", round(float(abs(row).max()), 3))

In [ ]:
import sys

sys.path.insert(0, str(REPO / "scripts"))
import check_prices

report = check_prices.check_hour(s, DAM_ID)
{k: report[k] for k in ("binding_rows", "rows_by_outcome", "system_price", "residual_by_kind", "points_cut_off_by_a_binding_contingency")}

## 8. The viewer

`viewer` writes a single HTML file that shows the network substation by substation, with two
snapshots side by side and every branch coloured by how it compares. **The file holds CEII** (names of
substations, buses and equipment): it stays inside the data folder, and you open it locally only. The
page keeps the current substation in the URL hash; open it with `?nohash` if the URL could ever leave
the machine.

In [ ]:
page = s.viewer(CRR_ID, DAM_ID)
print("written:", page.relative_to(s.data_dir), f"{page.stat().st_size / 1e6:.1f} MB")

## 9. The daily pull, the logs and what to look at

`scripts/daily_pull.py` runs at 07:00 every day through launchd (see `scripts/launchd/`). It fetches
every pulled product, lists the tracked ones, builds raw and core for whatever is new, and runs the
price check on the newest days. It writes:

- `data/logs/daily_pull.log`: one block per run, a line per product with counts and seconds, a line
  saying what is still **at risk** (EWS documents ERCOT offers that are not archived, and how many days
  before the oldest rolls off its window), and one summary line per day of the price check;
- `data/logs/last_run.json`: the last run's start and end, failures, the previous run's end, and the
  same risk figures. A macOS notification fires when anything failed.

If the machine slept through a run, launchd runs the job on wake; the Public API lookback is a month,
so prices heal themselves, and the EWS window is what the risk line watches. Older Public API days can
always be fetched by hand with `s.fetch(product, since=...)`.

Where to read next: `docs/assumptions.md` (every decision, its status and how it was checked),
`docs/datasets/*.md` (one note per dataset: what it is, how it is captured, what is parsed, the quirks),
`docs/out-network.md` (the network's conventions), and the reports under `data/reports/`.

In [ ]:
import json

status = REPO / "data" / "logs" / "last_run.json"
print(json.dumps(json.loads(status.read_text()), indent=1) if status.is_file() else "no production run on this machine yet")